# JAO Data Cleaning

### Columns to drop

* `refprog` – all 22 columns with missing values (2,016–30,839 missing out of 33,167 rows): `border_DE_DE_DK1_VH`, `border_DE_DK2_BigHub_DE`, `border_NL_DK1_COBRA`, `border_RO_RO_BG_VH`, `border_NL_NL_NO2_NorNed`, `border_DE_NO2_BigHub_DE`, `border_DE_DE_SE4_Baltic`, `border_PL_PL_SE4_SwePol`, `border_PL_LT_BigHub_PL`, `border_DK1_UK_Viking1`, `border_DK1_UK_Viking2`, `border_RO_MD`, `border_UA_MD`, `border_ES_MA_ESMA_link1`, `border_ES_MA_ESMA_link2`, `border_KS_ME`, `border_SI_HU`, `border_KS_AL`, `border_MK_KS`, `border_IT_MEMONITA2`, `border_ME_ITMONITA2`, `border_RS_KS`.
* `minmax_np` – all 18 columns with missing values (30,911 missing out of 33,263 rows): `min` and `max` columns of `DE_NO2_BigHub`, `NL_NO2_NorNed`, `DE_DK2_BigHub`, `DE_SE4_Baltic`, `PL_LT_BigHub`, `PL_SE4_SwePol`, `RO_BG_VH`, `NL_DK1_COBRA` and `DE_DK1_VH` (e.g. `minPL_LT_BigHub`, `maxPL_LT_BigHub`).
* `allocation_constraint` – `BE_export` (58,391 missing out of 58,391 rows) and `BE_import` (47,903 missing out of 58,391 rows).
* `d2cf` – no columns to drop.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import logging

sys.path.append(str(Path.cwd().parent.parent.parent))
from src.config import (
    DATA_DIR,
    DOWNLOAD_START_DATE,
    DOWNLOAD_END_DATE,
)
from src.utils.data_quality import (
    print_missing_values,
)

from src.utils.cleaning import clean_jao_datasets
from src.utils.file_io import load_jao_data, save_data, inspect_jao_files

In [2]:
date_range = f"{DOWNLOAD_START_DATE}_{DOWNLOAD_END_DATE}"
logging.basicConfig(level=logging.INFO)

In [3]:
data_jao = load_jao_data(DATA_DIR / "raw" / "jao" / date_range)

In [4]:
data_jao = clean_jao_datasets(data_jao)

## 1. Missing Values – Confirmation

After dropping the listed columns, none of the four datasets contains missing values (`d2cf` was complete from the start). This also removes the non-Core links, including the Polish `PL-LT` and `PL-SE4` connections; the gaps in the time index are handled later, during preprocessing.

In [5]:
for name, data in data_jao.items():
    print_missing_values(data=data, name=name)


--- [refprog] Missing values ---
No missing values.

--- [minmax_np] Missing values ---
No missing values.

--- [d2cf] Missing values ---
No missing values.

--- [allocation_constraint] Missing values ---
No missing values.


## 2. Conclusions

The cleaning decisions from the data quality audit have been applied to all four JAO datasets, and the check above confirms that no missing values remain. The whole-day and single-hour index gaps as well as the mixed resolution of `allocation_constraint` are left for the preprocessing step.

## 3. Saving Clean Data

In [6]:
save_data(data_jao, DATA_DIR / "clean" / "jao" / date_range)

INFO:src.utils.file_io:Saved clean/jao/2022-12-01_2026-09-16/refprog.parquet
INFO:src.utils.file_io:Saved clean/jao/2022-12-01_2026-09-16/minmax_np.parquet
INFO:src.utils.file_io:Saved clean/jao/2022-12-01_2026-09-16/d2cf.parquet
INFO:src.utils.file_io:Saved clean/jao/2022-12-01_2026-09-16/allocation_constraint.parquet


In [7]:
inspect_jao_files(DATA_DIR / "clean" / "jao" / date_range)


--- JAO (4 file/s) ------------------------------
  [FILE] data/clean/jao/2022-12-01_2026-09-16/refprog.parquet | shape: (33167, 65) | idx: 2022-12-01 00:00:00+01:00-2026-09-15 23:00:00+02:00
  [FILE] data/clean/jao/2022-12-01_2026-09-16/minmax_np.parquet | shape: (33263, 28) | idx: 2022-12-01 00:00:00+01:00-2026-09-16 23:00:00+02:00
  [FILE] data/clean/jao/2022-12-01_2026-09-16/d2cf.parquet | shape: (33163, 51) | idx: 2022-12-01 00:00:00+01:00-2026-09-15 23:00:00+02:00
  [FILE] data/clean/jao/2022-12-01_2026-09-16/allocation_constraint.parquet | shape: (58391, 2) | idx: 2022-12-01 00:00:00+01:00-2026-09-15 23:45:00+02:00
